# Prática MinIO com Boto3

Roteiro da seção 5 do `minio/README.md`: bucket, PUT, GET, listagem, presigned URL e exclusão.
O client é definido em `initBoto.ipynb`; aqui ele é repetido porque cada notebook tem seu próprio kernel.

In [ ]:
import socket
import boto3
from botocore.client import Config

# Dentro da rede mybridge (container do Spark) o MinIO responde pelo nome "minio";
# no host (VS Code no Arch) ele responde em localhost.
try:
    socket.gethostbyname('minio')
    endpoint = 'http://minio:9000'
except socket.gaierror:
    endpoint = 'http://localhost:9000'

s3 = boto3.client(
    's3',
    endpoint_url=endpoint,
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)
print('Endpoint:', endpoint)

## Buckets existentes

O `meu-bucket` foi criado antes pelo `mc mb`. Boto3 e `mc` falam a mesma API S3, então ele aparece aqui.

In [ ]:
print([b['Name'] for b in s3.list_buckets()['Buckets']])

## Criando um Bucket via API

Equivalente a `mc mb local/meu-novo-bucket`. A checagem evita o erro `BucketAlreadyOwnedByYou` ao rodar a célula de novo.

In [ ]:
BUCKET = 'meu-novo-bucket'

existentes = [b['Name'] for b in s3.list_buckets()['Buckets']]
if BUCKET not in existentes:
    s3.create_bucket(Bucket=BUCKET)
print([b['Name'] for b in s3.list_buckets()['Buckets']])

## Gravação (PUT)

Cria um arquivo local e envia ao bucket. O terceiro argumento é a **chave** do objeto.

In [ ]:
with open('localfile.txt', 'w') as f:
    f.write('Olá, MinIO! Primeiro objeto do data lake.\n')

s3.upload_file('localfile.txt', BUCKET, 'localfile.txt')

## Metadados do objeto

Diferente de um sistema de arquivos, o objeto carrega ETag, tamanho, tipo e data de modificação.

In [ ]:
meta = s3.head_object(Bucket=BUCKET, Key='localfile.txt')
for campo in ('ETag', 'ContentLength', 'ContentType', 'LastModified'):
    print(f'{campo}: {meta[campo]}')

## Listando Objetos em um Bucket

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET)
for obj in response.get('Contents', []):
    print(obj['Key'], obj['Size'], 'bytes')

## Leitura (GET)

In [ ]:
s3.download_file(BUCKET, 'localfile.txt', 'baixado_localfile.txt')

with open('baixado_localfile.txt') as f:
    print(f.read())

## Presigned URL

Link temporário que dá acesso ao objeto sem expor as credenciais.

In [ ]:
url = s3.generate_presigned_url(
    'get_object',
    Params={'Bucket': BUCKET, 'Key': 'localfile.txt'},
    ExpiresIn=3600  # segundos (1h)
)
print('URL temporária:', url)

## Testando com curl

Copie a URL acima e rode no terminal: `curl "<url>"`. Funciona sem usuário e senha, porque a autorização vai assinada na própria URL.

Sem assinatura, o acesso é negado (o bucket é privado):

In [ ]:
import urllib.request, urllib.error

try:
    urllib.request.urlopen(f'{endpoint}/{BUCKET}/localfile.txt')
except urllib.error.HTTPError as e:
    print('Sem assinatura:', e.code, e.reason)

print('Com presigned URL:', urllib.request.urlopen(url).read().decode())

## Deletando um Objeto

Rode por último, depois de conferir o objeto no console (http://localhost:9001).

In [ ]:
s3.delete_object(Bucket=BUCKET, Key='localfile.txt')
print(s3.list_objects_v2(Bucket=BUCKET).get('Contents', []))